In [1]:
# Automated Data Pipeline and Forensic Log Analyser

In [8]:
import requests, os
url = "https://api.github.com/repos/JayanGupta/Data_Science_Notes/contents/Sprint%201/data/daily_logs"

os.makedirs("daily_logs", exist_ok = True)

In [9]:
for f in requests.get(url).json():
  open("daily_logs/" + f['name'], "wb").write(requests.get(f['download_url']).content)

Q. How do we scan directory and filter hidden/ junk files?


In [14]:
import os
LOG_DIR = "/content/daily_logs"

def is_valid_logfile(filename):
  if filename.startswith(".") or filename.startswith("~"):
    return False

  return filename.lower().endswith(('.txt', '.json'))

In [15]:
all = os.listdir(LOG_DIR)

data_files = [os.path.join(LOG_DIR, f) for f in all if is_valid_logfile(f)]

data_files.sort()

In [16]:
print("total number of files: ", len(all))

total number of files:  74


In [17]:
print("total number of valid files", len(data_files))

total number of valid files 72


In [18]:
print("junk", len(all) - len(data_files))

junk 2


In [20]:
#print(data_files)

Q. How do we parser json file?

In [21]:
import json

In [82]:
from json.decoder import JSONDecodeError
def parser_json_file(filepath):
  try:
    with open(filepath, 'r', encoding='utf-8') as f:
      data = json.load(f)

    record = {
        'source': filepath,
        'ip_address': data['ip_address'],
        'status' : data['status_code']
    }

    return record, None

  except (json.JSONDecodeError, KeyError) as e:
    return None,  f"{type(e).__name__}: {e}"

In [83]:
sample = [f for f in data_files if f.endswith('.json')][5]

record = parser_json_file(sample)

print(record)

({'source': '/content/daily_logs/event_005.json', 'ip_address': '203.0.113.55', 'status': 301}, None)


In [84]:
#Q3 How do we safewly parse suingle text files?

In [85]:
def parse_text_file(filepath):
  try:
    with open(filepath, 'r', encoding='utf-8') as f:
      content = f.readline().strip()

    ip, status = content.split(',')
    record = {
        "source": filepath,
        "ip_address": ip.strip(),
        "status": status.strip()
    }

    return record, None

  except ValueError as e:
    return None, f"{type(e).__name__}: {e}"

In [86]:
sample = [f for f in data_files if f.endswith(".txt")][0]

record, error = parse_text_file(sample)

print(record, error)


{'source': '/content/daily_logs/access_040.txt', 'ip_address': '192.168.1.14', 'status': '503'} None


Q. How do we encapsulate this into clean dataparser class oop?

In [87]:
class DataParser:
  def __init__(self, filepath):
    self.filepath = filepath
    self.is_valid = True
    self.record = None
    self.error = None

  def parse(self):
    if self.filepath.lower().endswith(".json"):
      self.record, self.error = parser_json_file(self.filepath)

    else:
      self.record, self.error = parse_text_file(self.filepath)

    if self.error is not None:
      self.is_valid = False

    return self.record

In [88]:
bad = 0
for path in data_files:
  parser = DataParser(path)
  parser.parse()
  if not parser.is_valid:
    bad += 1

print("Bad files", {bad})

Bad files {7}


Q. How do we process all files and aggregate 5xx forensic anomalies?

In [91]:
from collections import Counter

clean_records = []

bad_records = []

error_counts = Counter()

for path in data_files:
  parser = DataParser(path)
  parser.parse()

  if parser.is_valid:
    clean_records.append(parser.record)

    if str(parser.record['status']).startswith("5"):
      error_counts[parser.record['ip_address']] += 1

  else:
    bad_records.append(path)


In [93]:
for ip, count in error_counts.most_common(5):
  print(f" IP: {ip:16s} | Errors: {count}")

 IP: 45.33.32.156     | Errors: 3
 IP: 198.51.100.23    | Errors: 3
 IP: 10.0.0.8         | Errors: 3
 IP: 192.168.1.14     | Errors: 2
 IP: 203.0.113.55     | Errors: 2


Q. How do we save and isolate corrupt files?

In [94]:
import shutil

In [116]:
OUTPUT_JSON = 'anomalies_ready.json'
with open(OUTPUT_JSON, 'w',encoding = 'utf-8') as f:
    json.dump(clean_records, f, indent=2)
print(f"[OUTPUT] Saved {len(clean_records)} valid records to '{OUTPUT_JSON}'")

[OUTPUT] Saved 65 valid records to 'anomalies_ready.json'


In [117]:
quarantine_dir = 'data/quarantine'
os.makedirs(quarantine_dir, exist_ok = True)

for bad_file in bad_records:
  dest = os.path.join(quarantine_dir, os.path.basename(bad_file))
  shutil.copy(bad_file, dest)

print(f"[Bad records ] isolated {len(bad_records)} corrupted files in '{quarantine_dir}/'")

[Bad records ] isolated 7 corrupted files in 'data/quarantine/'


Q How do we programmatically verify data integrity on disk

In [118]:
with open(OUTPUT_JSON, 'r', encoding = 'utf-8') as f:
  saved_data = json.load(f)
assert len(saved_data) == len(clean_records), "record count mismatch"
assert len(os.listdir(quarantine_dir)) == len(bad_records), "quarantine coutn mismatch"

print('[Verified] all prgrammatic verifciation assertation passed')
print(f'sample clean records: {saved_data[0]}')

[Verified] all prgrammatic verifciation assertation passed
sample clean records: {'source': '/content/daily_logs/access_040.txt', 'ip_address': '192.168.1.14', 'status': '503'}
